# Rate-Distortion Curves plotting from W&B Project

---

This notebook compares different runs with different activation functsion, output_padding, etc. for the ablation study of the paper.


#### Imports and global settings


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import wandb
from omegaconf import OmegaConf, DictConfig
from pathlib import Path
from pprint import pprint
import json
import numpy as np


# Quick ANSI color code shortcurts
r = "\033[31m"
y = "\033[33m"
g = "\033[32m"
b = "\033[34m"
e = "\033[0m"

api = wandb.Api()
# Project is specified by <entity/project-name>
entity: str = "cedric-leonard"
project: str = "SAR_DDC_FPGA"  # "SAR_DDC-RD-curve"
runs = api.runs(f"{entity}/{project}")
nb_all_runs = len(runs)
print(f"{b}Found {nb_all_runs} runs in project {entity}/{project}{e}")

verbose = False
save_loaded_project = True

ROOT_DIR = Path("..").resolve()
path_to_csv_results = ROOT_DIR / "results" / "plots" / "RD-curves_ablation_dataframe.csv"

# ── Shared color palette (Okabe-Ito, colorblind-safe) ────────────────────
C = json.load(open(ROOT_DIR / "notebooks" / "plots_colors.json"))

SAVE_FIGURES = True
FIGURE_FORMAT = "pdf"
PLOTS_DIR = ROOT_DIR / "results" / "plots" / "RD-curves"

# ── Color preview — VS Code shows swatches for hex literals ──────────────
# Tweak these hex values here, then copy them back to configs/plots_colors.json.
# ── output_padding group ─────────────────────────────────────────────────
_c_out_pad_with    = "#0072B2"  # output_padding.with_out_pad   (blue)
_c_out_pad_without = "#D55E00"  # output_padding.no_out_pad     (orange)
# ── activations group ────────────────────────────────────────────────────
_c_relu            = "#009E73"  # activations.relu              (green)
_c_gdn             = "#CC79A7"  # activations.gdn               (mauve)
# ── platforms group ──────────────────────────────────────────────────────
_c_fpga_dynamic    = "#009E73"  # platforms.fpga_dynamic        (green)
_c_fpga_idle       = "#85d4bc"  # platforms.fpga_idle           (pale green)
_c_gpu_dynamic     = "#0072B2"  # platforms.gpu_dynamic         (blue)
_c_gpu_idle        = "#7ab6d9"  # platforms.gpu_idle            (pale blue)
_c_cpu_dynamic     = "#D55E00"  # platforms.cpu_dynamic         (orange)
_c_cpu_idle        = "#ebb99a"  # platforms.cpu_idle            (pale orange)
# ── compare_gpu_fpga group ───────────────────────────────────────────────
_c_compare_gpu     = "#0072B2"  # compare_gpu_fpga.gpu          (blue)
_c_compare_fpga    = "#009E73"  # compare_gpu_fpga.fpga         (green)
# ── metrics group ────────────────────────────────────────────────────────
_c_metric_psnr     = "#0072B2"  # metrics.psnr                  (blue)
_c_metric_ssim     = "#009E73"  # metrics.ssim                  (green)
_c_metric_epd      = "#E69F00"  # metrics.epd                   (yellow-orange)
# ── architectures group ──────────────────────────────────────────────────
_c_arch_resshyp    = "#009E73"  # architectures.ResSHyp         (green)
_c_arch_shyp       = "#E69F00"  # architectures.SHyp            (yellow-orange)
# ── neutral / error bars ─────────────────────────────────────────────────
_c_errbar          = "#555555"  # neutral dark grey for error bar caps


#### Config loading


In [ ]:
run_name_list, id_list, run_output_dir_list,summary_list, config_list = [], [], [], [], []
data_name_list, model_name_list, lmbda_list, seed_list = [], [], [], []
no_output_padding_list, no_residual_blocks_list = [], []

for i, run in enumerate(runs):
    # Transform the wandb config in a DictConfig object (to be able to easily access nested keys)
    config = OmegaConf.create(run.config)

    run_name_list.append(run.name)  # .name is the human-readable name of the run.
    id_list.append(run.id)  # .id is the unique id of the run (used in logs and the URL).
    run_output_dir_list.append(OmegaConf.select(config, "paths.output_dir"))
    # .summary contains the output keys/values for metrics like accuracy. We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)

    if verbose and i == 0:
        print(f"Run N°{i} ({r}{run.id}{e}, '{run.name}') summary:")
        # print(f"BPP of Run N°{i} = {summary_list[i]['test/bpp']} and its config:")
        pprint(run.config)
        pprint(summary_list[i])
        # pprint(run.history())

    # Store the model name, lmbda (rate-distortion tradeoff) and the seed in separate lists (for easier access)
    dl_model_name = OmegaConf.select(config, "model.net._target_").split(".")[-1]
    prefix = run.name.split("_")[0]
    activation_name = OmegaConf.select(config, "model.net.activation")

    model_name_list.append(f"{dl_model_name}_{activation_name}")
    lmbda_list.append(OmegaConf.select(config, "lambda"))
    seed_list.append(OmegaConf.select(config, "seed"))
    # Get the dataset name from the config "data._target_" last part (after the last dot)
    data_name_list.append(
        OmegaConf.select(config, "data._target_").split(".")[-1]
    )  # e.g., EuroSATRGBDataModule or EuroSATSARDataModule
    no_output_padding_list.append(OmegaConf.select(config, "model.net.no_output_padding"))
    no_residual_blocks_list.append(OmegaConf.select(config, "model.net.no_residual_blocks"))

    #  Store the full config (just in case), remove special values that start with _.
    config_list.append({k: v for k, v in run.config.items()})  # if not k.startswith("_")})

# print(f"{len(model_name_list)=}, {len(lmbda_list)=}, {len(seed_list)=}, {len(config_list)=}")

# Create a dataframe with all the information
raw_runs_df = pd.DataFrame(
    {
        "id": id_list,
        "run_name": run_name_list,
        "data_name": data_name_list,
        "model_name": model_name_list,
        "lmbda": lmbda_list,
        "seed": seed_list,
        "no_output_padding": no_output_padding_list,
        "no_residual_blocks": no_residual_blocks_list,
        "summary": summary_list,
        "config": config_list,
        "run_output_dir": run_output_dir_list,
    }
)

# Set the index of the datafrane to be the run id
raw_runs_df.set_index("id", inplace=True)

# Save the dataframe to a csv file
if save_loaded_project:
    csv_file_path = f"{project}.csv"
    raw_runs_df.to_csv(csv_file_path)
    print(f"Dataframe saved to {b}{csv_file_path}{e}")
print(f"Project {b}{project}{e} contains {r}{nb_all_runs}{e} runs.")

## Filtering / Run cleaning

_This part is super **specific** and should probably be checked/changed everytime the goal of this script is different_


In [ ]:
def print_runs_info(df: pd.DataFrame, columns: list[str]):
    """Small utility function to print the run_name and the specified columns of a dataframe containing runs information."""
    for i, row in df.iterrows():
        # Don't start newline
        print(f"  - Run {i:<8} {row['run_name']:<50}:", end="")
        for col in columns:
            print(f" {r}{col}={row[col]}{e},", end="")
        print()


# ----- Remove runs run without a specific seed -----
# The GTs (ADAM-NOC and MERLIN) use seed = 42
list_accepted_seeds = [0, 1, 2, 3, 4, 5, 6]
print(f"{b}Removing runs whose seed is not in {list_accepted_seeds}...{e}")
print_runs_info(raw_runs_df[~raw_runs_df["seed"].isin(list_accepted_seeds)], ["seed"])
runs_df = raw_runs_df[
    raw_runs_df["seed"].isin(list_accepted_seeds)
]

# ----- Remove runs with specific tags in the config -----
list_tags_to_remove = ["debug", "crashed"]
for tag in list_tags_to_remove:
    print(f"{b}Removing runs with '{tag}' tags in the config...{e}")
    tag_runs = runs_df[runs_df["config"].apply(lambda x: tag in x["tags"])]
    if tag_runs.shape[0] > 0:
        print_runs_info(tag_runs, ["config"])
    runs_df = runs_df[~runs_df["config"].apply(lambda x: tag in x["tags"])]
    
# # ----- Remove runs that have residual blocks -----
# print(f"{b}Removing runs without residual blocks...{e}")
# no_residual_blocks_runs = runs_df[runs_df["no_residual_blocks"] == True]
# if no_residual_blocks_runs.shape[0] > 0:
#     print_runs_info(no_residual_blocks_runs, ["no_residual_blocks"])
# runs_df = runs_df[runs_df["no_residual_blocks"].isna() | (runs_df["no_residual_blocks"] == False)]

# # -----Remove runs that do not have a "test/bpp" key in the summary ("crashed run") -----
# print(f"{b}Removing runs without 'test/bpp' in the summary...{e}")
# no_bpp_runs = runs_df[~runs_df["summary"].apply(lambda x: "test/bpp" in x.keys())]
# if no_bpp_runs.shape[0] > 0:
#     # print([key for key in no_bpp_runs.iloc[0]['summary'].keys() if 'test/' in key])
#     print_runs_info(no_bpp_runs, ["summary"])
# runs_df = runs_df[runs_df["summary"].apply(lambda x: "test/bpp" in x.keys())]

# # ----- Filter out runs whose run_name does not start by "ResSHyp"-----
# print(f"{b}Removing runs whose run_name does not start by 'ResSHyp'...{e}")
# print_runs_info(runs_df[~runs_df["run_name"].str.startswith("ResSHyp")], [])
# runs_df = runs_df[runs_df["run_name"].str.startswith("ResSHyp")]

# # ----- Filter runs with "diverging"/"unnecessary" lmbda values -----
# print(f"{b}Removing runs with 'diverging'/'unnecessary' lmbda values (0.25, 0.5 and 1.5)...{e}")
# diverging_lmbda_runs = runs_df[runs_df["lmbda"].isin([0.25, 0.5, 1.5])]
# if diverging_lmbda_runs.shape[0] > 0:
#     print_runs_info(diverging_lmbda_runs, ["lmbda"])
# runs_df = runs_df[~runs_df["lmbda"].isin([0.25, 0.5, 1.5])]
# # runs_df = runs_df[runs_df["lmbda"].isin([0.1, 0.5, 1, 2, 5, 10, 25, 50, 100, 200, 500])]

# ---- Remove runs with "gdn1" activations -----
print(f"{b}Removing runs with 'gdn1' activations...{e}")
gdn1_runs = runs_df[runs_df["model_name"].str.contains("gdn1")]
if gdn1_runs.shape[0] > 0:
    print_runs_info(gdn1_runs, ["model_name"])
runs_df = runs_df[~runs_df["model_name"].str.contains("gdn1")]

# ----- Print the remaining runs ----- #
nb_filtered_runs = len(runs_df)
print(
    f"{r}{nb_all_runs - nb_filtered_runs}{e} runs were filtered out, {g}{nb_filtered_runs}{e} runs were kept."
)


In [ ]:
# --- Print unique model, lambda and seed per dataset ---
for i, name in enumerate(runs_df["data_name"].unique()):
    dataset_runs = runs_df[runs_df["data_name"] == name]
    unique_models = dataset_runs["model_name"].unique()
    unique_lmbda = dataset_runs["lmbda"].unique()
    unique_seeds = dataset_runs["seed"].unique()
    unique_no_output_padding = dataset_runs["no_output_padding"].unique()
    print(
        f"{r}{name}{e} ({g}{len(dataset_runs)}{e}) unique:\n"
        f"\t-  models: {g}{len(unique_models)}{e}, {unique_models}\n"
        f"\t- no_output_padding: {g}{len(unique_no_output_padding)}{e}, {unique_no_output_padding}\n"
        f"\t-  lambda: {g}{len(unique_lmbda)}{e}, {sorted(unique_lmbda.astype(float).tolist())}\n"
        f"\t-   seeds: {g}{len(unique_seeds)}{e}, {sorted(unique_seeds.astype(int).tolist())}\n"
    )
    expected_nb_runs = (
        len(unique_models) * len(unique_lmbda) * len(unique_seeds) * len(unique_no_output_padding)
    )
    if len(dataset_runs) != expected_nb_runs:
        print(
            f"   {y}Warning{e}: for all the unique values {b}{expected_nb_runs}{e} where expected! {b}{expected_nb_runs - len(dataset_runs)}{e} are missing.\n"
        )

# # Print all run hierarchically grouped by dataset, model, no_output_padding, lmbda, seed
# for dataset in runs_df["data_name"].unique():
#     dataset_runs = runs_df[runs_df["data_name"] == dataset]
#     print(f"{b}Dataset: {dataset}{e}")
#     for model in dataset_runs["model_name"].unique():
#         model_runs = dataset_runs[dataset_runs["model_name"] == model]
#         print(f"  {g}Model: {model}{e}")
#         for no_output_padding in model_runs["no_output_padding"].unique():
#             no_output_padding_runs = model_runs[
#                 model_runs["no_output_padding"] == no_output_padding
#             ]
#             print(f"    {b}No_output_padding: {no_output_padding}{e}")
#             for lmbda in sorted(no_output_padding_runs["lmbda"].unique().astype(float).tolist()):
#                 lmbda_runs = no_output_padding_runs[no_output_padding_runs["lmbda"] == lmbda]
#                 print(f"      {r}Lambda: {lmbda}{e}")
#                 for seed in sorted(lmbda_runs["seed"].unique().astype(int).tolist()):
#                     seed_runs = lmbda_runs[lmbda_runs["seed"] == seed]
#                     for run_id in seed_runs.index:
#                         print(f"        Seed: {seed} -> Run ID: {run_id}")

# Print the run_ids of the run with "data_name" == "EuroSATSARDataModule" and "model_name" == "mbt2018"
# print(
#     list(
#         runs_df[
#             (runs_df["data_name"] == "EuroSATSARDataModule")
#             & (runs_df["model_name"] == "mbt2018")
#         ].index
#     )
# )

#### Group and statistics


In [ ]:
verbose = False

# --- List of specific metrics to track for each model ---
metrics_of_interest = ["test/bpp"]
for metric in ["psnr", "ssim", "ms_ssim", "epd"]:  # "mse"
    metrics_of_interest.append(f"test/{metric}_noisy")   # reconstruction vs noisy (linear amplitude)
    metrics_of_interest.append(f"test/{metric}_merlin")
    metrics_of_interest.append(f"test/{metric}_adam_noc")
statistics_of_interest = ["mean", "min", "max", "std"]
metrics_to_track = []
for metric in metrics_of_interest:
    for stat in statistics_of_interest:
        metrics_to_track.append(f"{metric} {stat}")
print(f"Number of metrics to track: {len(metrics_to_track)}")

model_statistics = {}
# --- For each dataset ---
for data_name, df_data in runs_df.groupby("data_name"):
    # Append the list of metrics to track to the dictionanry
    model_statistics[data_name] = {}
    # --- For each model ---
    for (model_name, no_output_padding), df_model in df_data.groupby(
        ["model_name", "no_output_padding"]
    ):
        print(
            f"Processing dataset {data_name}, model {model_name} (no_output_padding={no_output_padding})..."
        )
        # Create an empty dataframe that will contain a column for each metric (test/psnr_noisy, test/bpp) and a row for each lmbda value
        lmbda_values = df_model["lmbda"].unique()
        stats_dataframe = pd.DataFrame(index=lmbda_values, columns=metrics_to_track)
        stats_dataframe.sort_index(inplace=True)  # Sort the index (only useful for nicer logs)

        # Compute a dictionary of statistics (mean, min, max, std) for 'test/psnr_noisy', 'test/psnr_merlin', 'test/bpp', etc.
        # in summary, for all the models having the same lmbda (therefore having different seeds).
        # Use .get(metric) so missing keys (e.g. from runs evaluated with older code) return None/NaN gracefully.
        stats = {}
        for lmbda, df_lmbda in df_model.groupby("lmbda"):
            for metric in metrics_of_interest:
                for stat in statistics_of_interest:
                    stats_dataframe.loc[(lmbda, f"{metric} {stat}")] = getattr(
                        df_lmbda["summary"].apply(lambda x: x.get(metric)),
                        stat,
                    )()

        if verbose:  # and model_name == "bmshj2018-factorized":
            pprint(stats_dataframe)

        model_name = f"{model_name}{'' if no_output_padding else '_out_pad'}"

        model_statistics[data_name][model_name] = stats_dataframe

# ── NaN diagnostic: identify lmbda values with missing metrics (cause discontinued curves) ──
# Check both mean (missing data entirely) and std (only 1 valid seed out of 6: metric stored
# as float NaN in W&B summary, which pandas .mean() skips but .std(ddof=1) needs ≥2 values).
CHECK_COLS = [
    "test/bpp mean", "test/psnr_merlin mean", "test/psnr_adam_noc mean",
    "test/bpp std",  "test/psnr_merlin std",  "test/psnr_adam_noc std",
]
print(f"\n{y}NaN check (missing values → discontinued curves / unreliable error bands):{e}")
any_nan = False
for data_name, models in model_statistics.items():
    for model_name, stats_df in models.items():
        for col in CHECK_COLS:
            if col not in stats_df.columns:
                continue
            nan_mask = stats_df[col].isna() | (stats_df[col].astype(str) == "nan")
            if nan_mask.any():
                missing = sorted(stats_df.index[nan_mask].astype(float).tolist())
                print(f"  {r}{data_name} / {model_name}{e}: '{b}{col}{e}' is NaN at λ = {missing}")
                any_nan = True
if not any_nan:
    print(f"  {g}No NaN values found — all curves should be continuous.{e}")


### Checking for NaN or missing values in all runs metrics

For each (model_name, lmbda) group that produced a NaN mean above, list the raw runs,
their run IDs, and which test/ metrics are missing from their W&B summary.

NOTE: metrics can be "missing" in two ways:
  (a) key absent from summary / value is None  → caught by `m not in summary or summary[m] is None`
  (b) key present but value is float NaN       → caught by the isnan check below
Both cases cause NaN in the computed statistics. Case (b) is the most common source of
NaN std when mean appears valid: if 1/6 seeds has a real value and 5/6 have float NaN,
mean = that single value (pandas skipna=True), but std = NaN (needs ≥2 valid values).

In [ ]:
import math

def is_nan_or_missing(summary: dict, metric: str) -> bool:
    """Return True if the metric is absent, None, or a float NaN in the W&B summary."""
    val = summary.get(metric)
    if val is None:
        return True
    try:
        return math.isnan(float(val))
    except (TypeError, ValueError):
        return False

NAN_CHECK_METRICS = [
    "test/bpp",
    "test/psnr_merlin",
    "test/psnr_adam_noc",
    "test/psnr_noisy",
    "test/ssim_merlin",
    "test/epd_merlin",
]

nan_run_ids = []  # collected for easy copy-paste into update_wandb_runs.py

print(f"{y}Runs with NaN / missing metrics:{e}\n")
print(f"  {'Run ID':<12} {'Name':<55} {'λ':>6}  {'seed':>4}  {'no_out_pad':>10}  Missing metrics")
print(f"  {'-'*12} {'-'*55} {'-'*6}  {'-'*4}  {'-'*10}  ---------------")

for data_name, models in model_statistics.items():
    for model_key, stats_df in models.items():
        for col in CHECK_COLS:
            if col not in stats_df.columns:
                continue
            nan_mask = stats_df[col].isna() | (stats_df[col].astype(str) == "nan")
            if not nan_mask.any():
                continue
            nan_lmbdas = stats_df.index[nan_mask].astype(float).tolist()

            # Find the matching raw runs in runs_df
            # model_key naming: "{model_name}{'' if no_output_padding else '_out_pad'}"
            # Reverse: '_out_pad' suffix → no_output_padding=False
            has_out_pad_suffix = model_key.endswith("_out_pad")
            # Strip the suffix to get back the original model_name stored in runs_df
            base_model_name = model_key[: -len("_out_pad")] if has_out_pad_suffix else model_key

            candidate_runs = runs_df[
                (runs_df["data_name"] == data_name)
                & (runs_df["model_name"] == base_model_name)
                & (runs_df["no_output_padding"] == (not has_out_pad_suffix))
                & (runs_df["lmbda"].isin(nan_lmbdas))
            ]

            for run_id, run_row in candidate_runs.iterrows():
                summary = run_row["summary"]
                # Check both absent/None AND float NaN stored in summary
                missing = [m for m in NAN_CHECK_METRICS if is_nan_or_missing(summary, m)]
                nan_run_ids.append(run_id)
                flag = f"{r}MISSING/NaN{e}" if missing else f"{g}present{e}"
                print(
                    f"  {run_id:<12} {run_row['run_name']:<55} {run_row['lmbda']:>6.0f}"
                    f"  {str(run_row['seed']):>4}  {str(not has_out_pad_suffix):>10}"
                    f"  {flag}: {missing}"
                )
            break  # Only need to report once per (model, set of NaN lambdas)

nan_run_ids = list(dict.fromkeys(nan_run_ids))  # deduplicate, preserve order
print(f"\n{y}→ {len(nan_run_ids)} runs to re-evaluate.{e}")
print(f"\nTo re-run {b}update_wandb_runs.py{e} for only these runs, add this line")
print(f"inside the main() loop (after 'matching_runs = ...' filter lines):\n")
print(f'    matching_runs = [r for r in matching_runs if r.id in {nan_run_ids}]')


## Save into a CSV file

In [ ]:
# for each model append a single csv file with the statistics
with open(path_to_csv_results, "w") as f:
    f.write("data_name,model_name,lmbda," + ",".join(metrics_to_track) + "\n")
    for data_name, models in model_statistics.items():
        for model_name, stats_df in models.items():
            for lmbda, row in stats_df.iterrows():
                f.write(
                    f"{data_name},{model_name},{lmbda},"
                    + ",".join([str(x) for x in row.values])
                    + "\n"
                )
print(f"Statistics saved to {b}{path_to_csv_results}{e}")

# Plots RD-curves 

In [ ]:
# Color/style dicts from shared palette (configs/plots_colors.json)
colors = {
    "Output padding": C["output_padding"]["with_out_pad"],
    "Without output padding": C["output_padding"]["no_out_pad"],
}
activation_names = {"gdn": "GDN", "relu": "ReLU", "gdn1": "GDN1"}
linestyles = {"GDN": "dotted", "ReLU": "solid"}  # "gdn1": "dashed",


def plot_psnr_bpp_curve(ax, bpp_mean: float, psnr_mean: float, linestyle: str, color: str):
    ax.plot(
        bpp_mean,
        psnr_mean,
        "o",
        linestyle=linestyle,
        color=color,
        # label=f"{model_name} {ref}",
    )


def add_psnr_error_bands(ax, bpp_mean: float, psnr_min: float, psnr_max: float, color: str):
    ax.fill_between(
        bpp_mean,
        psnr_max,
        psnr_min,
        alpha=0.2,
        color=color,
    )


def add_bpp_error_bars(ax, bpp_mean: float, psnr_mean: float, bpp_std: float):
    ax.errorbar(
        bpp_mean,
        psnr_mean,
        xerr=bpp_std,
        fmt="none",
        ecolor=C["metrics"]["error_bars"],
        capsize=2,  # Set the size of the vertical lines at the end of the error bars
    )


def add_lambda_labels(ax, model_name: str, ref: str):
    # Add text labels for each lmbda value
    for model in model_statistics["TSXSSCDataModule"]:
        if model == model_name:
            for lmbda, row in model_statistics["TSXSSCDataModule"][model_name].iterrows():
                ax.text(
                    row["test/bpp mean"] + 0.05,
                    row[f"test/{ref} mean"],
                    f"{int(lmbda)}",
                    ha="center",
                    va="bottom",
                )


def add_legend(ax, colors: dict[str, str], linestyles: dict[str, str], fontsize: int = 8):
    for key, value in colors.items():
        ax.plot([], [], color=value, label=f"{key}")
    for key, value in linestyles.items():
        ax.plot([], [], color=C["metrics"]["error_bars"], linestyle=value, label=f"{key}")
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), fontsize=fontsize)

#### Plot the rate distorion curves


In [ ]:
ref = "psnr_merlin"
fig, ax = plt.subplots(figsize=(9, 6))

for j, model_name in enumerate(model_statistics["TSXSSCDataModule"]):
    print(
        f"Plotting {model_statistics['TSXSSCDataModule'][model_name].shape[0]} lmbda values for model {model_name}..."
    )
    color = colors["Output padding"] if "out_pad" in model_name else colors["Without output padding"]
    # Set linestyle based on activation
    linestyle = linestyles[activation_names[model_name.split("_")[1]]]

    bpp_mean = model_statistics["TSXSSCDataModule"][model_name]["test/bpp mean"].astype(float)
    bpp_std = model_statistics["TSXSSCDataModule"][model_name]["test/bpp std"].astype(float)
    psnr_mean = model_statistics["TSXSSCDataModule"][model_name][f"test/{ref} mean"].astype(float)
    psnr_std = model_statistics["TSXSSCDataModule"][model_name][f"test/{ref} std"].astype(float)
    psnr_min = model_statistics["TSXSSCDataModule"][model_name][f"test/{ref} min"].astype(float)
    psnr_max = model_statistics["TSXSSCDataModule"][model_name][f"test/{ref} max"].astype(float)
    
    plot_psnr_bpp_curve(ax, bpp_mean, psnr_mean, linestyle, color)
    add_psnr_error_bands(ax, bpp_mean, psnr_min, psnr_max, color)
    # add_psnr_error_bands(ax, bpp_mean, psnr_mean - psnr_std, psnr_mean + psnr_std, color)
    add_bpp_error_bars(ax, bpp_mean, psnr_mean, bpp_std)


# add_lambda_labels(ax, "ResidualScaleHyperpriorPatched_gdn_out_pad", ref)
# add_lambda_labels(ax, "ResidualScaleHyperpriorPatched_relu", ref)

# ax.set_title("TSXSSCDataModule data compression models comparison to MERLIN")
ax.set_xlabel("Bit-rate [bpp]")
ax.set_ylabel("PSNR [dB]")
ax.grid()
add_legend(ax, colors, linestyles, fontsize=12)
plt.tight_layout()


if SAVE_FIGURES:
    plt.savefig(PLOTS_DIR / f"RD-curves_ablation_PSNR-MERLIN.{FIGURE_FORMAT}", bbox_inches="tight")
plt.show()

In [ ]:
# Multi-metric RD curve grid — comparison to MERLIN reference
# Each subplot shows a different quality metric against BPP.

MERLIN_METRICS = [
    ("psnr_merlin", "PSNR vs MERLIN [dB]"),
    ("ssim_merlin", "SSIM vs MERLIN"),
    ("epd_merlin",  "EPD vs MERLIN"),
]


def plot_rd_curves_grid(
    statistics: dict,
    data_name: str,
    quality_metrics: list,
    prefix: str = "test",
    ncols: int = 3,
    suptitle: str = "",
) -> None:
    """Plot a grid of RD curves, one subplot per quality metric.

    Args:
        statistics:      model_statistics dict  (data_name → model_name → stats_df)
        data_name:       key in statistics, e.g. "TSXSSCDataModule"
        quality_metrics: list of (metric_suffix, y_label)
        prefix:          metric prefix used in column names, e.g. "test"
        ncols:           number of columns in the subplot grid
        suptitle:        overall figure title
    """
    n = len(quality_metrics)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5 * nrows), squeeze=False)

    for ax, (metric_suffix, y_label) in zip(axes.flat, quality_metrics):
        ax.set_xlabel("Bit-rate [bpp]")
        ax.set_ylabel(y_label)
        ax.set_title(y_label)
        ax.grid(True, alpha=0.3)
        
        if "epd" in metric_suffix:
            # limit the Y axis to max 2
            ax.set_ylim(0, 1.2)

        for model_name, stats_df in statistics.get(data_name, {}).items():
            color = colors["Output padding"] if "out_pad" in model_name else colors["Without output padding"]
            linestyle = linestyles[activation_names[model_name.split("_")[1]]]

            bpp_col     = f"{prefix}/bpp mean"
            bpp_std_col = f"{prefix}/bpp std"
            q_col       = f"{prefix}/{metric_suffix} mean"
            q_min_col   = f"{prefix}/{metric_suffix} min"
            q_max_col   = f"{prefix}/{metric_suffix} max"

            if bpp_col not in stats_df.columns or q_col not in stats_df.columns:
                continue

            bpp_arr = stats_df[bpp_col].astype(float).to_numpy()
            q_arr   = stats_df[q_col].astype(float).to_numpy()
            valid   = ~(np.isnan(bpp_arr) | np.isnan(q_arr))

            if valid.sum() == 0:
                continue

            sort_idx = np.argsort(bpp_arr[valid])
            bpp_s    = bpp_arr[valid][sort_idx]
            q_s      = q_arr[valid][sort_idx]

            ax.plot(bpp_s, q_s, "o", linestyle=linestyle, color=color, markersize=4)

            if bpp_std_col in stats_df.columns:
                bpp_std = stats_df[bpp_std_col].astype(float).to_numpy()[valid][sort_idx]
                ax.errorbar(bpp_s, q_s, xerr=bpp_std, fmt="none", ecolor=C["metrics"]["error_bars"], capsize=2, alpha=0.5)

            if q_min_col in stats_df.columns and q_max_col in stats_df.columns:
                q_min = stats_df[q_min_col].astype(float).to_numpy()[valid][sort_idx]
                q_max = stats_df[q_max_col].astype(float).to_numpy()[valid][sort_idx]
                ax.fill_between(bpp_s, q_min, q_max, alpha=0.15, color=color)

    # # Hide unused axes
    # for ax in list(axes.flat)[n:]:
    #     ax.set_visible(False)
    for ax in axes.flat:
        add_legend(ax, colors, linestyles, fontsize=10)
    fig.suptitle(suptitle or f"RD curves — {data_name}", fontsize=13)
    plt.tight_layout()
    
    if SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"RD-curves_ablation_subplots.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()


plot_rd_curves_grid(
    model_statistics,
    "TSXSSCDataModule",
    MERLIN_METRICS,
    suptitle="TSXSSCDataModule — compression performance vs MERLIN reference",
)


# Subset representativeness check: `test/` vs `test_sub500/`

Compare the same metrics computed on the full test set (`test/`) and on the 500-patch subset (`test_sub500/`).
A well-representative subset should produce curves and numbers that nearly overlap.

In [ ]:
# Build per-prefix statistics tables for test/ and test_sub500/.
# Requires that both prefixes are present in the W&B summary (i.e. after the retest script finishes).

PREFIXES = ["test", "test_sub500"]
METRICS_COMPARE = ["bpp", "psnr_merlin", "psnr_adam_noc", "psnr_noisy"]
STATS_COMPARE = ["mean", "std"]

import numpy as np

def safe_float(x):
    """Convert a W&B summary value to float, mapping None/Infinity strings to NaN."""
    try:
        v = float(x)
        return v if np.isfinite(v) else float("nan")
    except (TypeError, ValueError):
        return float("nan")

# Only runs that have BOTH prefixes available can be compared.
def has_both_prefixes(summary: dict) -> bool:
    return "test/bpp" in summary and "test_sub500/bpp" in summary

runs_both = runs_df[runs_df["summary"].apply(has_both_prefixes)]
print(
    f"{g}{len(runs_both)}{e} / {len(runs_df)} runs have both test/ and test_sub500/ metrics."
)
if len(runs_both) == 0:
    print(f"{r}No runs have both prefixes yet — re-run this cell once the retest script finishes.{e}")

# Build a dict: prefix -> {data_name -> {model_name -> stats_df}}
prefix_statistics: dict = {p: {} for p in PREFIXES}

for prefix in PREFIXES:
    metrics_list = [f"{prefix}/{m}" for m in METRICS_COMPARE]
    cols = [f"{m} {s}" for m in metrics_list for s in STATS_COMPARE]

    for data_name, df_data in runs_both.groupby("data_name"):
        prefix_statistics[prefix][data_name] = {}
        for (model_name, no_output_padding), df_model in df_data.groupby(
            ["model_name", "no_output_padding"]
        ):
            lmbda_values = sorted(df_model["lmbda"].unique().astype(float).tolist())
            stats_df = pd.DataFrame(index=lmbda_values, columns=cols, dtype=float)

            for lmbda, df_lmbda in df_model.groupby("lmbda"):
                for metric in metrics_list:
                    for stat in STATS_COMPARE:
                        # Convert to float first to avoid '-Infinity' string errors
                        numeric_series = df_lmbda["summary"].apply(
                            lambda x: safe_float(x.get(metric))
                        )
                        stats_df.loc[lmbda, f"{metric} {stat}"] = getattr(numeric_series, stat)()

            key = f"{model_name}{'' if no_output_padding else '_out_pad'}"
            prefix_statistics[prefix][data_name][key] = stats_df

print("Statistics built for prefixes:", list(prefix_statistics.keys()))


#### Side-by-side RD curves: full set vs sub500

In [ ]:
# For each quality metric, plot full-set curve vs sub500 curve for all models.
# Encoding:
#   color     → output_padding  (blue = with_out_pad, red = no_out_pad)
#   linestyle → activation      (solid = ReLU, dotted = GDN)
#   marker    → prefix          (o = full test set, x = sub500)

from matplotlib.lines import Line2D

# Per-prefix: only the marker changes (and a slight alpha reduction for sub500)
PREFIX_STYLES = {
    "test":        {"marker": "o", "alpha": 1.0, "markersize": 4, "markeredgewidth": 1.0},
    "test_sub500": {"marker": "x", "alpha": 0.7, "markersize": 6, "markeredgewidth": 1.5},
}

for ref_metric in ["psnr_merlin"]: # "psnr_adam_noc", "psnr_noisy"
    fig, ax = plt.subplots(figsize=(10, 6))
    ref_name = ref_metric.split('_', 1)[1].upper()

    for model_key in prefix_statistics["test"].get("TSXSSCDataModule", {}):
        # Visual encoding from model identity
        color     = colors["Output padding"] if "out_pad" in model_key else colors["Without output padding"]
        linestyle = linestyles[activation_names[model_key.split("_")[1]]]

        for prefix, style in PREFIX_STYLES.items():
            stats = prefix_statistics[prefix].get("TSXSSCDataModule", {}).get(model_key)
            if stats is None:
                continue
            bpp_col  = f"{prefix}/bpp mean"
            psnr_col = f"{prefix}/{ref_metric} mean"
            if bpp_col not in stats.columns or psnr_col not in stats.columns:
                continue

            bpp  = stats[bpp_col].astype(float)
            psnr = stats[psnr_col].astype(float)
            valid = bpp.notna() & psnr.notna()
            if valid.sum() == 0:
                continue

            bpp_arr  = bpp[valid].to_numpy()
            psnr_arr = psnr[valid].to_numpy()
            sort_idx = np.argsort(bpp_arr)

            ax.plot(
                bpp_arr[sort_idx],
                psnr_arr[sort_idx],
                color=color,
                linestyle=linestyle,
                marker=style["marker"],
                alpha=style["alpha"],
                markersize=style["markersize"],
                markeredgewidth=style["markeredgewidth"],
            )

    legend_elems = [
        Line2D([0], [0], color=colors["Output padding"], label="with output_padding"),
        Line2D([0], [0], color=colors["Without output padding"],   label="no output_padding"),
        Line2D([0], [0], color=C["metrics"]["error_bars"], linestyle=linestyles["ReLU"], label="ReLU activation"),
        Line2D([0], [0], color=C["metrics"]["error_bars"], linestyle=linestyles["GDN"],  label="GDN activation"),
        Line2D([0], [0], color=C["metrics"]["error_bars"], linestyle="-", marker="o", markersize=4,
               markeredgewidth=1.0, label="full test set"),
        Line2D([0], [0], color=C["metrics"]["error_bars"], linestyle="-", marker="x", markersize=6,
               markeredgewidth=1.5, label="sub500"),
    ]
    ax.legend(handles=legend_elems, loc="lower right", fontsize=9)
    ax.set_title(f"Full set vs sub500 — {ref_name}")
    ax.set_xlabel("Bit-rate [bpp]")
    ax.set_ylabel("PSNR [dB]")
    ax.grid(True)
    plt.tight_layout()
    
    if SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"RD-curves_subtest_{ref_name}.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()


#### Numerical comparison table: mean metric difference (full − sub500)

For each model group and lambda, compute `test/ − test_sub500/` for key metrics.
Values near zero mean the subset is representative.

In [ ]:
DIFF_METRICS = ["bpp", "psnr_merlin", "psnr_adam_noc", "psnr_noisy"]

diff_rows = []
for model_key in prefix_statistics["test"].get("TSXSSCDataModule", {}):
    stats_full = prefix_statistics["test"]["TSXSSCDataModule"].get(model_key)
    stats_sub  = prefix_statistics["test_sub500"]["TSXSSCDataModule"].get(model_key)
    if stats_full is None or stats_sub is None:
        continue
    lmbdas = sorted(set(stats_full.index) & set(stats_sub.index))
    for lmbda in lmbdas:
        row = {"model": model_key, "lambda": lmbda}
        for m in DIFF_METRICS:
            full_val = stats_full.loc[lmbda, f"test/{m} mean"]
            sub_val  = stats_sub.loc[lmbda,  f"test_sub500/{m} mean"]
            try:
                row[f"Δ {m}"] = float(full_val) - float(sub_val)
            except (TypeError, ValueError):
                row[f"Δ {m}"] = float("nan")
        diff_rows.append(row)

diff_df = pd.DataFrame(diff_rows).set_index(["model", "lambda"])

# Style: highlight large absolute differences in red
def color_diff(val):
    try:
        v = float(val)
        if abs(v) > 1:
            return "background-color: #ffcccc"
        if abs(v) > 0.5:
            return "background-color: #fff3cc"
    except (TypeError, ValueError):
        pass
    return ""

print("Δ = full test set − sub500  (ideally close to 0)")
display(diff_df.style.format("{:.4f}", na_rep="—").applymap(color_diff))


#### Scatter plot: full set vs sub500 (one dot per run)

Each dot is one (model, lambda, seed) run. If the subset is representative, all dots should lie on the diagonal.

In [ ]:
SCATTER_METRICS = ["bpp", "psnr_merlin", "ssim_merlin"]

fig, axes = plt.subplots(1, len(SCATTER_METRICS), figsize=(5 * len(SCATTER_METRICS), 5))

for ax, metric in zip(axes, SCATTER_METRICS):
    full_vals, sub_vals, dot_colors = [], [], []

    for _, row in runs_both.iterrows():
        s = row["summary"]
        full_v = s.get(f"test/{metric}")
        sub_v  = s.get(f"test_sub500/{metric}")
        if full_v is None or sub_v is None:
            continue
        full_vals.append(float(full_v))
        sub_vals.append(float(sub_v))
        # Color by model type
        mn = row["model_name"]
        nop = row["no_output_padding"]
        if "relu" in mn and nop:
            dot_colors.append(colors["Without output padding"])
        elif "relu" in mn:
            dot_colors.append(C["activations"]["relu"])
        elif nop:
            dot_colors.append(colors["Output padding"])
        else:
            dot_colors.append(C["activations"]["gdn"])

    if not full_vals:
        ax.set_title(f"{metric}\n(no data yet)")
        continue

    vmin = min(min(full_vals), min(sub_vals))
    vmax = max(max(full_vals), max(sub_vals))
    margin = (vmax - vmin) * 0.05
    diag = [vmin - margin, vmax + margin]

    ax.scatter(full_vals, sub_vals, c=dot_colors, alpha=0.6, s=18, zorder=3)
    ax.plot(diag, diag, "k--", lw=1, label="y = x")
    ax.set_xlabel(f"test/{metric}")
    ax.set_ylabel(f"test_sub500/{metric}")
    ax.set_title(metric)
    ax.grid(True, alpha=0.4)

# Shared legend
from matplotlib.patches import Patch
legend_patches = [
    Patch(color=colors["Without output padding"],   label="ReLU + no_output_padding"),
    Patch(color=C["activations"]["relu"],            label="ReLU + output_padding"),
    Patch(color=colors["Output padding"],            label="GDN + no_output_padding"),
    Patch(color=C["activations"]["gdn"],             label="GDN + output_padding"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.04))
fig.suptitle("Per-run scatter: full test set vs sub500", fontsize=13)
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(PLOTS_DIR / f"Scatter_full-vs-sub_testset_subplot.{FIGURE_FORMAT}", bbox_inches="tight")
plt.show()
